<h2>Formating of DB-Data<h2>

In [1]:
from hilfsfunktionen.formating import get_unchanged_table, get_formated_table, get_table_with_keys, create_n_sentences
df_collection = {}
df_collection["unchanged_table"] = get_unchanged_table()
df_collection["formated_table"] = get_formated_table()
df_collection["formated_table_with_keys"] = get_table_with_keys(df_collection["formated_table"])
df_collection["n_sentences"] = {}
for n in [1, 5, 20, 50, 100]:
    df_collection["n_sentences"][f"{n}"] = create_n_sentences(df_collection["formated_table_with_keys"], n)


<h2>Model Creation<h2>

In [2]:
from hilfsfunktionen.model_training import create_many_models
from pathlib import Path

models, categories = create_many_models(models = {}, n_sentences_tokenized = df_collection["n_sentences"], sentences_tokenized = df_collection["n_sentences"]["20"],base_dir = Path.cwd(), verbose = False)



Model Creation started.
Model Creation finished.


<h2>Durchschnittliche Satzvektoren berechnen<h2>

In [4]:
from hilfsfunktionen.sentence_vectors import create_average_sentence_vectors
aver_sent_vec = []
for category in categories:
    print(f"(-------------Vector calculation of category: {category.upper()}-------------)")
    for model in models[category]:
        aver_sent_vec.append([create_average_sentence_vectors(model, df_collection["formated_table_with_keys"], base_dir = Path.cwd(), verbose=False),model[0]])
    print(f"(-------------Finished Vector calculation of category: {category.upper()}-------------)")


(-------------Vector calculation of category: COMPARISON-------------)
(-------------Finished Vector calculation of category: COMPARISON-------------)
(-------------Vector calculation of category: SENTENCES-------------)
(-------------Finished Vector calculation of category: SENTENCES-------------)
(-------------Vector calculation of category: VECTOR-------------)
(-------------Finished Vector calculation of category: VECTOR-------------)
(-------------Vector calculation of category: WINDOWSIZE-------------)
(-------------Finished Vector calculation of category: WINDOWSIZE-------------)
(-------------Vector calculation of category: EPOCH-------------)
(-------------Finished Vector calculation of category: EPOCH-------------)
(-------------Vector calculation of category: NEGATIVE-------------)
(-------------Finished Vector calculation of category: NEGATIVE-------------)
(-------------Vector calculation of category: SAMPLE-------------)
(-------------Finished Vector calculation of catego

<h2>Vektoren-Test Textbasiert<h2>

In [5]:
from hilfsfunktionen.basic_vector_tests import run_full_quality_check
from hilfsfunktionen.key_vector_tests import run_full_quality_check_key
from hilfsfunktionen.cosine_tests import run_full_vector_test
    
def run_all_tests(model,verbose):
    run_full_quality_check(model, verbose)
    run_full_quality_check_key(model, verbose, df_with_keys = df_collection["formated_table_with_keys"])
    run_full_vector_test(model, verbose, df_with_keys = df_collection["formated_table_with_keys"])

In [ ]:
for category in categories:
    print(f"(-------------Test of category: {category.upper()}-------------)\n")
    for model in models[category]:
        run_all_tests(model,False)


(-------------Test of category: COMPARISON-------------)


################################################
        AUTOMATISCHER W2V QUALITY CHECK
               Model Parameters:       
Vektorraumgröße = 150 Fenstergröße  = 4 Algorithmus = CBOW
         Epochen = 30 Anzahl der Sätze = 10000 Länge der Sätze = 340
################################################

Semantische Erwartungspaare:  4 von 5 Erfolgreich
Länder-Nachbarn:  3 von 3 Erfolgreich
Alters-Nachbarn:  3 von 3 Erfolgreich
Gehalts-Nachbarn:  3 von 3 Erfolgreich
Credit_Score-Nachbarn:  3 von 3 Erfolgreich
Churn-Nachbarn:  2 von 2 Erfolgreich

################################################
        AUTOMATISCHER W2V KEY CHECK
################################################

Key-Struktur: 
 Analysis for key_1 
Matching pairs in the 0th most similar: (3/11) 
Matching pairs in the 1th most similar: (4/11) 
Matching pairs in the 2th most similar: (2/11) 
Matching pairs in the 3th most similar: (4/11) 
Matching pairs in the 4t

<h2>Visualisierung<h2>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
import math

#Alle Wörter:
def get_filtered_words(model, filter, prefix="key_"):
    words = list(model.wv.key_to_index.keys())
    if filter:
        return [w for w in words if not w.startswith(prefix)]
    else:
        return words


def visualize_all_models(models_list, max_cols=2):
    """
    Visualisiert alle Modelle in einem automatisch angepassten Grid
    
    Args:
        models_list: Liste von [model, info_dict] Paaren
        max_cols: Maximale Anzahl an Spalten
    """
    n_models = len(models_list)
    
    # Berechne optimale Grid-Größe
    n_cols = min(max_cols, n_models)
    n_rows = math.ceil(n_models / n_cols)
    
    fig, axes = plt.subplots(
        n_rows, 
        n_cols,
        figsize=(6*n_cols, 5*n_rows),
        dpi=120,
        constrained_layout=True
    )
    
    # Falls nur ein Plot, mache axes zu einer Liste
    if n_models == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes.flatten()
    elif n_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    # Definiere gemeinsame Wörter für alle Visualisierungen
    common_words = [
        'Balance_Cluster_1', 'Balance_Cluster_5', 'Balance_Cluster_10',
        'Age_Young_Adults', 'Age_Middle_aged', 'Age_Young_Seniors',
        'Salarie_Low', 'Salarie_Average', 'Salarie_High',
        'Credit_Score_Poor', 'Credit_Score_Good', 'Credit_Score_Excellent',
        'Tenure_0', 'Tenure_24', 'Tenure_60'
    ]
    
    
    
    for idx, (model, info) in enumerate(models_list):
        common_words = get_filtered_words(model, True)
        if idx < len(axes):
            ax = axes[idx]
            
            # Verfügbare Wörter im aktuellen Modell
            available_words = [w for w in common_words if w in model.wv.key_to_index]
            
            if len(available_words) >= 5:
                # t-SNE Berechnung
                vectors = np.array([model.wv[w] for w in available_words])
                tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(available_words)-1))
                reduced = tsne.fit_transform(vectors)
                
                # Plot
                scatter = ax.scatter(reduced[:, 0], reduced[:, 1], alpha=0.7, s=60)
                
                # Annotationen
                for i, word in enumerate(available_words):
                    ax.annotate(word, xy=(reduced[i, 0], reduced[i, 1]), 
                               fontsize=6, alpha=0.8, 
                               xytext=(5, 5), textcoords='offset points')
                
                # Titel mit Modell-Informationen
                title = f"Model {idx+1}\n"
                keys = ['Vector Size', 'Window', 'Algorithmus', 'Kleinste Worthäufigkeit', 'Workers', 'Epochs', 'Satzanzahl', 'Satzlänge']
                
                # Erste Zeile: erste 3 Parameter
                first_line = []
                for key in keys[:4]:
                    if key in info:
                        first_line.append(f"{key}: {info[key]}")
                if first_line:
                    title += ", ".join(first_line) + "\n"
                
                # Zweite Zeile: restliche Parameter
                second_line = []
                for key in keys[4:]:
                    if key in info:
                        second_line.append(f"{key}: {info[key]}")
                if second_line:
                    title += ", ".join(second_line)
                
                title = title.rstrip('\n')  # Falls zweite Zeile leer ist
                ax.set_title(title, fontsize=10)
                ax.grid(True, alpha=0.3)
            
            else:
                ax.text(0.5, 0.5, f"Model {idx+1}\nNicht genug Wörter", 
                       ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f"Model {idx+1}", fontsize=10)
    
    # Verstecke leere Subplots
    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    print("Erste Darstellung erfolgreich.")



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def visualize_model_comparison(models_list):
    """
    Zeigt Ähnlichkeits-Matrizen für alle Modelle im Vergleich
    Immer zwei Grafiken nebeneinander
    """
    n_models = len(models_list)
    
    # Immer zwei Spalten (Grafiken nebeneinander)
    n_cols = 2
    n_rows = (n_models + 1) // 2  # Aufrunden für ungerade Anzahl
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    
    # Falls nur eine Zeile, axes als 2D-Array behandeln
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Gemeinsame Wörter für Vergleich
    comparison_words = [
        'Balance_Cluster_1', 'Balance_Cluster_5', 
        'Age_Young_Adults', 'Age_Middle_aged',
        'Salarie_Low', 'Salarie_High',
        'Credit_Score_Poor', 'Credit_Score_Excellent'
    ]
    
    for idx, (model, info) in enumerate(models_list):
        # Position berechnen
        row = idx // n_cols
        col = idx % n_cols
        
        ax = axes[row, col]
        
        # Verfügbare Wörter
        available_words = [w for w in comparison_words if w in model.wv.key_to_index]
        
        if len(available_words) >= 3:
            # Ähnlichkeitsmatrix berechnen
            vectors = np.array([model.wv[w] for w in available_words])
            similarity_matrix = cosine_similarity(vectors)
            
            # Heatmap plotten
            sns.heatmap(similarity_matrix, 
                       xticklabels=available_words,
                       yticklabels=available_words,
                       annot=True, fmt='.2f',
                       cmap='coolwarm', center=0,
                       ax=ax, cbar=False)
            
            # Titel mit Modell-Informationen
            title = f"Model {idx+1}\n"
            keys = ['Vector Size', 'Window', 'Algorithmus', 'Kleinste Worthäufigkeit', 'Workers', 'Epochs', 'Satzanzahl', 'Satzlänge']
            
            for i, key in enumerate(keys):
                if key in info:
                    title += f"{key}: {info[key]}, "
                    if i >= 2:  # Nach den ersten 3 Einträgen Umbruch
                        title += f"\n"
            title = title.rstrip(', ')  
            ax.set_title(title, fontsize=10)
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        
        else:
            ax.text(0.5, 0.5, "Nicht genug Wörter\nfür Vergleich", 
                   ha='center', va='center', transform=ax.transAxes)
            
            # Titel auch für Modelle ohne Vergleich
            title = f"Model {idx+1}\n"
            keys = ['Vector Size', 'Window', 'Algorithmus', 'Kleinste Worthäufigkeit', 'Workers', 'Epochs', 'Satzanzahl', 'Satzlänge']
            
            for i, key in enumerate(keys):
                if key in info:
                    title += f"{key}: {info[key]}, "
                    if i >= 2:
                        title += f"\n"
            title = title.rstrip(', ')
            ax.set_title(title, fontsize=10)
    
    # Leere Subplots ausblenden, falls ungerade Anzahl an Modellen
    for idx in range(n_models, n_rows * n_cols):
        row = idx // n_cols
        col = idx % n_cols
        axes[row, col].set_visible(False)
    
    plt.tight_layout()
    plt.show()

# Modell-Metriken anzeigen
def print_model_metrics(models_list):
    """Gibt Metriken für alle Modelle aus"""
    print(f"{'Model':<10} {'Vector Size':<12} {'Window':<8} {'Algorithmus':<12} {'Vocab Size':<12}")
    print("-" * 70)
    
    for idx, (model, info) in enumerate(models_list):
        vocab_size = len(model.wv.key_to_index)
        vector_size = info.get('Vector Size', 'N/A')
        window = info.get('Window', 'N/A')
        algorithmus = info.get('Algorithmus', 'N/A')
        
        print(f"{idx+1:<10} {vector_size:<12} {window:<8} {algorithmus:<12} {vocab_size:<12}")

In [ ]:
def visualize_model(model_name):
    visualize_all_models(models[model_name])
    print_model_metrics(models[model_name])
    visualize_model_comparison(models[model_name])

In [ ]:
visualize_model("sentences")

In [ ]:
visualize_model("vector")

In [ ]:
visualize_model("windosize")

In [ ]:
visualize_model("epoch")